In [ ]:
# 📦 Install dependencies
!pip install rdflib gradio networkx matplotlib

In [ ]:
# 📚 Imports and Namespaces
import rdflib
from rdflib import Graph, RDF, Namespace, Literal
import networkx as nx
import matplotlib.pyplot as plt
import gradio as gr
from io import BytesIO

KG = Namespace("http://genetics.org/")
g = Graph()
g.bind("kg", KG)


In [ ]:
# 🧠 Add RDF Triples
g.add((KG.SNP001, KG.gene, KG.BRCA1))
g.add((KG.SNP001, KG.associatedDisease, KG.BreastCancer))
g.add((KG.SNP002, KG.gene, KG.APOE))
g.add((KG.SNP002, KG.associatedDisease, KG.Alzheimers))

In [ ]:
# 🔄 RDF to NetworkX Graph
def rdf_to_nx(rdf_graph):
    G = nx.DiGraph()
    for s, p, o in rdf_graph:
        s = str(s).split("/")[-1]
        p = str(p).split("/")[-1]
        o = str(o).split("/")[-1] if isinstance(o, rdflib.URIRef) else str(o)
        G.add_edge(s, o, label=p)
    return G


In [ ]:
# 🎯 Draw function
def draw_kg_graph():
    G = rdf_to_nx(g)
    pos = nx.spring_layout(G, seed=42)
    edge_labels = nx.get_edge_attributes(G, 'label')
    fig, ax = plt.subplots(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, node_color='lightgreen', node_size=2500, edge_color='gray', font_size=10, ax=ax)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='purple', ax=ax)
    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    return buf


In [ ]:
# 💬 Q&A logic
def simple_qa(question):
    q = question.lower()
    if "brca" in q or "breast" in q:
        return "SNP001 in BRCA1 gene is associated with Breast Cancer"
    elif "apoe" in q or "alzheimer" in q:
        return "SNP002 in APOE gene is associated with Alzheimer's"

    return "Sorry, I couldn't find relevant information."

with gr.Blocks() as demo:
    gr.Markdown("## 🧬 Genetic Data & SNPs Knowledge Graph & Q&A")
    btn = gr.Button("Show Graph")
    img = gr.Image(type="filepath")
    q_input = gr.Textbox(label="Ask a Question")
    a_output = gr.Textbox(label="Answer")

    btn.click(fn=draw_kg_graph, inputs=[], outputs=img)
    q_input.submit(fn=simple_qa, inputs=q_input, outputs=a_output)

demo.launch()
